# Data Audit and EDA

Before analysing anything, every table in the database is checked for missing values, duplicate rows, and whether its ID column is actually unique. Each finding is then followed up to understand its cause, and the decision taken is recorded.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

conn = sqlite3.connect('../olist.db')

# Load every table into a dictionary so we can loop over them later
tables = {}

tables['orders'] = pd.read_sql_query("SELECT * FROM orders", conn)
tables['customers'] = pd.read_sql_query("SELECT * FROM customers", conn)
tables['order_items'] = pd.read_sql_query("SELECT * FROM order_items", conn)
tables['products'] = pd.read_sql_query("SELECT * FROM products", conn)
tables['payments'] = pd.read_sql_query("SELECT * FROM payments", conn)
tables['reviews'] = pd.read_sql_query("SELECT * FROM reviews", conn)
tables['sellers'] = pd.read_sql_query("SELECT * FROM sellers", conn)
tables['geolocation'] = pd.read_sql_query("SELECT * FROM geolocation", conn)
tables['category_translation'] = pd.read_sql_query("SELECT * FROM category_translation", conn)

conn.close()

# Print how big each table is
for name in tables:
    df = tables[name]
    print(name, "-", len(df), "rows,", len(df.columns), "columns")

orders - 99441 rows, 8 columns
customers - 99441 rows, 5 columns
order_items - 112650 rows, 7 columns
products - 32951 rows, 9 columns
payments - 103886 rows, 5 columns
reviews - 99224 rows, 7 columns
sellers - 3095 rows, 4 columns
geolocation - 1000163 rows, 5 columns
category_translation - 71 rows, 2 columns


In [2]:
# Check every table for missing values, one at a time

for name in tables:
    df = tables[name]
    
    print("=" * 60)
    print(name.upper(), "-", len(df), "rows")
    print("=" * 60)
    
    missing = df.isnull().sum()
    
    # Only show columns that actually have missing values
    missing = missing[missing > 0]
    
    if len(missing) == 0:
        print("No missing values")
    else:
        for column in missing.index:
            count = missing[column]
            percent = round(count / len(df) * 100, 2)
            print(column, ":", count, "missing (", percent, "%)")
    print()

ORDERS - 99441 rows
order_approved_at : 160 missing ( 0.16 %)
order_delivered_carrier_date : 1783 missing ( 1.79 %)
order_delivered_customer_date : 2965 missing ( 2.98 %)

CUSTOMERS - 99441 rows
No missing values

ORDER_ITEMS - 112650 rows
No missing values

PRODUCTS - 32951 rows
product_category_name : 610 missing ( 1.85 %)
product_name_lenght : 610 missing ( 1.85 %)
product_description_lenght : 610 missing ( 1.85 %)
product_photos_qty : 610 missing ( 1.85 %)
product_weight_g : 2 missing ( 0.01 %)
product_length_cm : 2 missing ( 0.01 %)
product_height_cm : 2 missing ( 0.01 %)
product_width_cm : 2 missing ( 0.01 %)

PAYMENTS - 103886 rows
No missing values

REVIEWS - 99224 rows
review_comment_title : 87656 missing ( 88.34 %)
review_comment_message : 58247 missing ( 58.7 %)

SELLERS - 3095 rows
No missing values

GEOLOCATION - 1000163 rows
No missing values

CATEGORY_TRANSLATION - 71 rows
No missing values



In [3]:
# Check for duplicate rows in each table

for name in tables:
    df = tables[name]
    duplicate_rows = df.duplicated().sum()
    print(name, ":", duplicate_rows, "duplicate rows")

print()
print("-" * 60)
print()

# Check whether each table's ID column is actually unique

print("orders - order_id unique?", tables['orders']['order_id'].nunique() == len(tables['orders']))
print("customers - customer_id unique?", tables['customers']['customer_id'].nunique() == len(tables['customers']))
print("products - product_id unique?", tables['products']['product_id'].nunique() == len(tables['products']))
print("sellers - seller_id unique?", tables['sellers']['seller_id'].nunique() == len(tables['sellers']))
print("reviews - review_id unique?", tables['reviews']['review_id'].nunique() == len(tables['reviews']))

orders : 0 duplicate rows
customers : 0 duplicate rows
order_items : 0 duplicate rows
products : 0 duplicate rows
payments : 0 duplicate rows
reviews : 0 duplicate rows
sellers : 0 duplicate rows
geolocation : 261831 duplicate rows
category_translation : 0 duplicate rows

------------------------------------------------------------

orders - order_id unique? True
customers - customer_id unique? True
products - product_id unique? True
sellers - seller_id unique? True
reviews - review_id unique? False


In [4]:
# FINDING 1: are the missing delivery dates just non-delivered orders?

orders = tables['orders']

print("Order status for orders missing a delivery date:")
missing_delivery = orders[orders['order_delivered_customer_date'].isnull()]
print(missing_delivery['order_status'].value_counts())

print()
print("Delivered orders that are STILL missing a delivery date:")
delivered = orders[orders['order_status'] == 'delivered']
print(delivered['order_delivered_customer_date'].isnull().sum())

Order status for orders missing a delivery date:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

Delivered orders that are STILL missing a delivery date:
8


In [5]:
# FINDING 2: how bad is the review duplication?

reviews = tables['reviews']

print("Total review rows:", len(reviews))
print("Unique review_id:", reviews['review_id'].nunique())
print("Unique order_id:", reviews['order_id'].nunique())
print()
print("Orders with more than one review:",
      len(reviews) - reviews['order_id'].nunique())

Total review rows: 99224
Unique review_id: 98410
Unique order_id: 98673

Orders with more than one review: 551


In [6]:
# FINDING 3: are the 610 products missing all four fields together?

products = tables['products']

no_category = products['product_category_name'].isnull()
no_name_length = products['product_name_lenght'].isnull()

print("Missing category:", no_category.sum())
print("Missing name length:", no_name_length.sum())
print("Missing BOTH:", (no_category & no_name_length).sum())

Missing category: 610
Missing name length: 610
Missing BOTH: 610


In [7]:
# FINDING 4: do those 610 products actually get ordered?

order_items = tables['order_items']

bad_product_ids = products[no_category]['product_id']
affected = order_items[order_items['product_id'].isin(bad_product_ids)]

print("Order items involving products with no category:", len(affected))
print("As a share of all order items:",
      round(len(affected) / len(order_items) * 100, 2), "%")

Order items involving products with no category: 1603
As a share of all order items: 1.42 %


In [8]:
# FINDING 2 follow-up: why does one review_id appear on several orders?

reviews = tables['reviews']

# Count how many rows each review_id appears on
review_counts = reviews['review_id'].value_counts()

# Keep only the ones that appear more than once
repeated_reviews = review_counts[review_counts > 1]

print("Review IDs attached to more than one order:", len(repeated_reviews))
print()

# Look at one example in detail
example_id = repeated_reviews.index[0]
example_rows = reviews[reviews['review_id'] == example_id]

print("Example review_id:", example_id)
print(example_rows[['review_id', 'order_id', 'review_score']])

Review IDs attached to more than one order: 789

Example review_id: c444278834184f72b1484dfe47de7f97
                              review_id                          order_id  \
2952   c444278834184f72b1484dfe47de7f97  df56136b8031ecd28e200bb18e6ddb2e   
43897  c444278834184f72b1484dfe47de7f97  566f53fc7d36fa366f7e221468121877   
95774  c444278834184f72b1484dfe47de7f97  8557dabbdacec1a9e250f5e70afe1eab   

       review_score  
2952              5  
43897             5  
95774             5  


In [9]:
# Attach customer and purchase time to every order in a repeated review

orders = tables['orders']
customers = tables['customers']

multi = reviews[reviews['review_id'].isin(repeated_reviews.index)]

multi = multi.merge(orders[['order_id', 'customer_id', 'order_purchase_timestamp']],
                    on='order_id', how='left')
multi = multi.merge(customers[['customer_id', 'customer_unique_id']],
                    on='customer_id', how='left')

multi['order_purchase_timestamp'] = pd.to_datetime(multi['order_purchase_timestamp'])

# Question 1: how many different customers per review?
customers_per_review = multi.groupby('review_id')['customer_unique_id'].nunique()

# Question 2: how far apart were the orders, in minutes?
first_order_time = multi.groupby('review_id')['order_purchase_timestamp'].min()
last_order_time = multi.groupby('review_id')['order_purchase_timestamp'].max()
gap_minutes = (last_order_time - first_order_time).dt.total_seconds() / 60

total = len(repeated_reviews)

print("Repeated reviews checked:", total)
print()
print("All orders belong to ONE customer:", (customers_per_review == 1).sum())
print("All orders placed within 1 minute:", (gap_minutes <= 1).sum())
print("All orders placed within 24 hours:", (gap_minutes <= 1440).sum())

Repeated reviews checked: 789

All orders belong to ONE customer: 789
All orders placed within 1 minute: 730
All orders placed within 24 hours: 748


**Result:** all 789 repeated reviews belong to a single customer, and 93% of them cover
orders placed within one minute of each other. One checkout with several sellers produces
several order IDs, and the customer's single review is recorded against each. This is the
same split-basket pattern found in the SQL notebook, confirmed here from a different
direction.

## Findings and decisions

| # | Finding | Cause | Decision |
|---|---|---|---|
| 1 | 2,965 orders have no delivery date | Orders that were shipped, cancelled, unavailable, or still processing | Excluded from delivery and satisfaction analysis by filtering to delivered orders |
| 2 | 8 orders are marked delivered but have no delivery date | Inconsistent records | Excluded — no delivery outcome to measure |
| 3 | 551 orders have more than one review row | Duplicate review records | Keep only the earliest review per order |
| 4 | 789 reviews are shared across more than one order | Split baskets: one review per checkout, recorded against every order in it | Kept. Affects ~1.6% of review rows; noted as a limitation in the delivery analysis |
| 5 | 610 products missing category and all descriptive fields | Product records that were never fully populated | Labelled "unknown" in category analysis rather than dropped, since they account for 1.42% of items sold |
| 6 | 261,831 duplicate rows in geolocation | Repeated coordinates for the same zip prefix | No action — table not used |
| 7 | Review comment text missing on 59–88% of reviews | Customers leaving a score without writing anything | No action — only the numeric score is used |
| 8 | Missing approval and carrier dates on some orders | Order lifecycle stages not reached | No action — these columns are not used |

**Missing review scores are never imputed.** An order without a review is a customer who
chose not to leave one, which is different information from a low score. Filling it in
would invent opinions that don't exist.

## Tables used in this project

| Used | Why |
|---|---|
| orders | purchase and delivery dates |
| customers | links orders to people, and gives state |
| order_items | order value and item counts |
| reviews | satisfaction scores |
| sellers | seller ranking by state |
| products, category_translation | category breakdown |

| Not used | Why |
|---|---|
| geolocation | this project has no map component |
| payments | payment method and instalments aren't part of any question this project asks |